## Class Creation - Data Puller

In [1]:
# CLASS CREATION - STOCK_DATA_PULLER

!pip install alpaca-py

from alpaca.data.historical import StockHistoricalDataClient
from alpaca.data.requests import StockBarsRequest
from alpaca.data.timeframe import TimeFrame, TimeFrameUnit
from datetime import datetime, timedelta

class AlpacaDataPuller:
    '''
    Pulls Stock Data from Alpaca API.
    symbols: List of stock symbols
    api_key: Alpaca account API Key
    secret_key: Alpaca account Secret Key
    bar_size: In Minutes - how long you want each bar - ie open to close length
    '''
    def __init__ (
        self, 
        symbols: list[str], 
        api_key: str,
        secret_key: str,
        timeframe_amount: int = 5,
        timeframe_unit: str = "minute"
    ):

        self.symbols = symbols
        self.api_key = api_key
        self.secret_key = secret_key
        self.timeframe_amount = timeframe_amount
        self.timeframe_unit = timeframe_unit

        # INITIALIZE ALPACA CLIENT
        self.client = StockHistoricalDataClient(
            api_key,
            secret_key
        )

    def get_timeframe(self, amount=None, unit=None):
        """
        Converts amount/unit inputs into an Alpaca TimeFrame object.
        """

        if amount is None:
            amount = self.timeframe_amount

        if unit is None:
            unit = self.timeframe_unit

        unit = unit.lower()

        unit_map = {
            "minute": TimeFrameUnit.Minute,
            "min": TimeFrameUnit.Minute,
            "hour": TimeFrameUnit.Hour,
            "day": TimeFrameUnit.Day,
            "week": TimeFrameUnit.Week,
        }

        if unit not in unit_map:
            raise ValueError("timeframe_unit must be: minute, hour, day, or week")

        return TimeFrame(amount, unit_map[unit])

    # Pulls stock data for a single Symbol
    def pull_symbol_data(self, symbol, start= None, end= None, timeframe_amount = None, timeframe_unit = None):
        '''
        Pulls stock data for given symbol 'XYZ'
        start & end format: (YYYY, MM, DD)
        If start/ end blank, defaults to last 10 years
        '''

        # Checks for start and end input - 10 years as default
        if end is None:
            end_date = datetime.today()
        else:
            end_date = datetime(*end)

        if start is None:
            start_date = end_date - timedelta(days=365 * 10)
        else:
            start_date = datetime(*start)

        request = StockBarsRequest(
        symbol_or_symbols=symbol,
        timeframe=self.get_timeframe(timeframe_amount, timeframe_unit),
        start=start_date,
        end=end_date,
        )

        bars = self.client.get_stock_bars(request)
        df = bars.df
        return df


    # Uses previous method to combine multiple symbols into a dictionary
    def pull_symbols_data(self, symbols=None, start=None, end=None, timeframe_amount = None, timeframe_unit = None):
        '''
        Pulls data for multiple symbols (list) into a dictionary
        '''
        # If NO symbols are passed, default to class symbols
        if symbols is None:
            symbols = self.symbols

        data = {}

        for symbol in symbols:
            data[symbol] = self.pull_symbol_data(symbol,start,end, timeframe_amount, timeframe_unit)

        return data
    

    # Establish Baselines based on Quartiles

    # def create_baselines(self, zero_thresh = .02):
    #     '''
    #     Establishes 5 classes based on 4 quartiles and zero threshold
    #     zero threshold: 0 +/- percent of avg threshold for 'Flat' classification
    #     default zero threshold: 2% of average movement per given timeframe
    #     '''


## SPY Example using AlpacaDataPuller

In [2]:
## TEST CELL

# REQUIRED INPUTS
API_KEY = "PKEN63LDIFPD43FZN5JQXXXHXV"
SECRET_KEY = "3pmEQxuC9R5vGL4y6YSfDMWq9fgSM8nBZae8QGHyoE3C"
SYMBOLS = ['SPY']

# PULL SPY HISTORICALS INTO DICTIONARY BY TIME UNIT

puller_hrly = AlpacaDataPuller(SYMBOLS,API_KEY,SECRET_KEY, timeframe_amount=1, timeframe_unit="hour")

puller_daily = AlpacaDataPuller(SYMBOLS,API_KEY,SECRET_KEY, timeframe_amount=1, timeframe_unit="day")

puller_weekly = AlpacaDataPuller(SYMBOLS,API_KEY,SECRET_KEY, timeframe_amount=1, timeframe_unit="week")

SPY_dict = {
    'hourly': puller_hrly.pull_symbols_data(),
    'daily': puller_daily.pull_symbols_data(),
    'weekly': puller_weekly.pull_symbols_data()
}


In [3]:
#print(SPY_dict["hourly"]["SPY"].head())
#print(SPY_dict["daily"]["SPY"].head())
print(SPY_dict["weekly"]["SPY"].head(10))

                                    open      high     low     close  \
symbol timestamp                                                       
SPY    2016-06-20 04:00:00+00:00  208.82  210.8700  202.72  203.2425   
       2016-06-27 04:00:00+00:00  201.59  210.4900  198.65  209.9208   
       2016-07-04 04:00:00+00:00  208.95  212.9400  207.06  212.6500   
       2016-07-11 04:00:00+00:00  213.19  217.0125  212.95  215.8300   
       2016-07-18 04:00:00+00:00  215.97  217.3700  215.63  217.2400   
       2016-07-25 04:00:00+00:00  217.00  217.5400  215.62  217.1200   
       2016-08-01 04:00:00+00:00  217.19  218.2300  214.25  218.1800   
       2016-08-08 04:00:00+00:00  218.40  218.9400  217.23  218.4600   
       2016-08-15 04:00:00+00:00  218.89  219.5000  217.02  218.5400   
       2016-08-22 04:00:00+00:00  218.27  219.6000  216.25  217.2900   

                                       volume  trade_count        vwap  
symbol timestamp                                              